<!-- cabecera-entorno -->
## Antes de empezar

**Clase 7 · Construir con IA: agentes, skills y ecosistema** — Bloque 3 · Reto. Este cuaderno lo
recorre **su equipo solo**, leyendo: cada tarea trae la explicación y lo que necesita antes de
pedirle nada. El profesor circula por el salón resolviendo dudas. Es el entregable de la clase.

**Este reto no necesita Node.js, ni una cuenta, ni internet.** Solo Python y el entorno del curso.
Instalar un CLI de IA es una **recomendación**, no un requisito, y **no se evalúa**: si su equipo
no tiene ninguno instalado, hace el reto completo igual que todo el mundo.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |
| `PermissionError` al escribir un archivo | El cuaderno se abrió desde una carpeta donde no puede escribir | Manual, problema 6 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

CARPETA_SKILLS = Path(".gemini/skills")
CARPETA_SKILLS.mkdir(parents=True, exist_ok=True)
print("Sus skills se van a escribir en:", CARPETA_SKILLS.resolve())

# Clase 7 · Bloque 3 — Reto

## Su ecosistema de 3 skills

Universidad Cooperativa de Colombia · Analítica de Datos · Momento 2

---

**Equipo:** _escriba aquí los nombres_

**Dataset del proyecto:** _nombre del archivo y de dónde salió_

---

## Consigna

Diseñe y construya el ecosistema de **3 skills** que su proyecto del semestre va a necesitar.

No skills genéricas. Skills para **su** dataset y **sus** preguntas de investigación.

La diferencia entre un ecosistema y una colección de archivos sueltos es que el ecosistema cubre
**fases distintas** del trabajo. Tres skills que hacen casi lo mismo no son un ecosistema: son un
skill mal partido en tres.

## Por qué esto importa ahora y no en la clase 12

Los skills que escriba hoy los va a usar en las clases 8, 9, 10 y 11. Si los escribe bien, cada
uno le ahorra trabajo cuatro veces. Si los escribe el día antes de la sustentación, no le ahorran
nada.

Esto no es un ejercicio de calentamiento. Es infraestructura.

## Qué se verifica hoy, y qué no

Este cuaderno trae un verificador, y conviene ser exacto sobre lo que puede y lo que no:

| Sí comprueba | No comprueba |
|--------------|--------------|
| Que el plan tenga forma de ecosistema y cubra dos fases | Que sean los tres skills que su proyecto de verdad necesita |
| Que los tres archivos estén bien armados | Que los skills sean buenos |
| Que el skill 1 nombre las columnas reales de su CSV | Que las instrucciones sean las correctas para su pregunta |
| Que la salida que usted espera cumpla el contrato que su propio skill declara | Que la salida real del modelo se le parezca |
| Que ninguno de sus skills dispare una bandera roja de seguridad | Si el skill es útil |

**Ninguna tarea de hoy se comprueba contra una respuesta correcta**, porque su proyecto es suyo y
este cuaderno no lo conoce. Lo que se comprueba es la forma, y la forma es donde están la mayoría
de los errores.

Lea el `README.md` de esta carpeta si necesita el detalle de la consigna y de la entrega.

In [ ]:
# Verificador de las siete tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
# Solo usa la libreria estandar de Python: no necesita internet ni Node.js.
import hashlib
import re
import unicodedata
from pathlib import Path

_RESULTADOS = {}

_CLAVES = ["R1", "R2", "R3", "R4", "R5", "R6", "R7"]

_PISTAS = {
    "R1": "El plan es una lista de tres tuplas: (nombre, fase, que sale). El nombre va en formato de carpeta: minusculas, guiones, sin tildes ni espacios. La fase es una de preparacion, analisis o comunicacion, y tienen que aparecer al menos dos distintas. Si no puede describir en cinco palabras que sale del skill, el skill todavia no existe: vuelva a pensarlo antes de abrir un archivo.",
    "R2": "Vuelva a la celda que escribe el archivo y complete lo que falte: frontmatter con los tres campos, instrucciones, bloque '## Formato' con sus secciones '### ' y bloque '## Reglas' con dos reglas como minimo, una de ellas un limite. Despues vuelva a ejecutar esa celda: el verificador lee el disco, no la celda.",
    "R3": "Lo mismo que el skill 1. Si le aparece que sigue el texto de la plantilla, es que quedo algun corchete sin reemplazar.",
    "R4": "Lo mismo. Este es el que puede quedar sin probar, pero no sin escribir.",
    "R5": "Su skill tiene que nombrar por lo menos dos de las columnas reales que escribio en la Tarea 0, con el nombre exacto del CSV. Si no las nombra, ese skill serviria igual para cualquier dataset del mundo, y los genericos ya existen.",
    "R6": "Escriba la salida que espera de su skill 1, con las mismas secciones que su bloque '## Formato' promete y con cifras inventadas donde haga falta. Si una seccion no le sale, el problema no es la prediccion: es que su formato no dice lo suficiente.",
    "R7": "El auditor detecta las banderas 1, 2 y 3. Si su propio skill dispara alguna, revise que le esta pidiendo leer o mandar. Y el veredicto lo escribe usted: si un companero de otro equipo se encontrara este archivo, ¿lo instalaria? Diga si o no, y por que."
}

_ESPERADO = {}


# --- Utilidades ------------------------------------------------------------

def _normalizar(texto):
    """Minusculas, sin tildes y sin puntuacion. Para comparar sin falsos negativos."""
    plano = unicodedata.normalize("NFKD", str(texto).lower())
    plano = "".join(c for c in plano if not unicodedata.combining(c))
    return "".join(c if c.isalnum() or c.isspace() else " " for c in plano)


def _firma(valor):
    """Reduce una respuesta a un texto reproducible, sin importar como se escribio."""
    if isinstance(valor, (list, tuple, set)):
        piezas = sorted(_normalizar(v).strip() for v in valor)
        return "coleccion|" + "|".join(p for p in piezas if p)
    if isinstance(valor, str):
        return "texto|" + _normalizar(valor).strip()
    try:
        return f"numero|{round(float(valor), 4)}"
    except (TypeError, ValueError):
        return f"otro|{valor!r}"


def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]


# --- Lectura de un SKILL.md ------------------------------------------------

_MARCADORES_DE_PLANTILLA = (
    "tu codigo aqui", "nombre legible del skill", "una frase que diga cuando",
    "instrucciones especificas", "la estructura exacta de la salida",
    "primera seccion de la salida", "segunda seccion de la salida",
    "una restriccion concreta", "un limite de alcance",
    "escriba aqui", "escribe aqui", "por definir aqui",
)


def leer_skill(ruta):
    """Lee un SKILL.md y lo parte en frontmatter, instrucciones y secciones.

    Devuelve un diccionario. Si el archivo no existe o el frontmatter esta roto,
    la clave "error" trae el diagnostico y las demas vienen vacias. Es la misma
    lectura que hace una herramienta real: si esto falla, el skill nunca se activa.
    """
    resultado = {"ruta": str(ruta), "error": None, "meta": {}, "cuerpo": "", "texto": ""}
    archivo = Path(ruta)
    if not archivo.exists():
        resultado["error"] = f"no existe el archivo {ruta}"
        return resultado
    if archivo.name != "SKILL.md":
        resultado["error"] = (f"el archivo se llama '{archivo.name}': tiene que llamarse "
                              f"exactamente SKILL.md, en mayusculas")
        return resultado
    texto = archivo.read_text(encoding="utf-8")
    resultado["texto"] = texto
    lineas = texto.splitlines()
    if not lineas or lineas[0].strip() != "---":
        resultado["error"] = ("la primera linea del archivo no es '---': sin frontmatter la "
                              "herramienta lee el archivo como texto plano y el skill nunca "
                              "se activa")
        return resultado
    cierre = None
    for i, linea in enumerate(lineas[1:], start=1):
        if linea.strip() == "---":
            cierre = i
            break
    if cierre is None:
        resultado["error"] = ("el frontmatter abre con '---' y nunca cierra: hacen falta los "
                              "tres guiones de cierre en su propia linea")
        return resultado
    for linea in lineas[1:cierre]:
        if not linea.strip():
            continue
        if ":" not in linea:
            continue
        clave, _, valor = linea.partition(":")
        resultado["meta"][clave.strip().lower()] = valor.strip()
    resultado["cuerpo"] = "\n".join(lineas[cierre + 1:])
    return resultado


def _bloque_de_seccion(cuerpo, titulo):
    """Devuelve el texto de una seccion '## titulo' hasta el siguiente '## '."""
    patron = re.compile(r"^##\s+" + titulo + r"\s*$", re.IGNORECASE | re.MULTILINE)
    encontrado = patron.search(cuerpo)
    if not encontrado:
        return None
    resto = cuerpo[encontrado.end():]
    siguiente = re.search(r"^##\s+", resto, re.MULTILINE)
    return resto[:siguiente.start()] if siguiente else resto


def secciones_de_formato(cuerpo):
    """Las secciones que el bloque '## Formato' declara. Es el contrato de salida."""
    bloque = _bloque_de_seccion(cuerpo, "Formato")
    if bloque is None:
        return []
    secciones = [linea.strip().lstrip("#").strip()
                 for linea in bloque.splitlines() if linea.strip().startswith("###")]
    return [s for s in secciones if s]


def reglas_declaradas(cuerpo):
    """Las vinetas del bloque '## Reglas'."""
    bloque = _bloque_de_seccion(cuerpo, "Reglas")
    if bloque is None:
        return []
    reglas = []
    for linea in bloque.splitlines():
        limpia = linea.strip()
        if limpia.startswith(("- ", "* ")):
            reglas.append(limpia[2:].strip())
        elif re.match(r"^\d+\.\s", limpia):
            reglas.append(re.sub(r"^\d+\.\s*", "", limpia).strip())
        elif reglas and limpia and not limpia.startswith("#"):
            reglas[-1] = (reglas[-1] + " " + limpia).strip()
    return [r for r in reglas if r]


# --- Reglas de forma (no imprimen: devuelven listas de faltas) -------------

def faltas_de_skill(lectura):
    """Las fallas de FORMA de un SKILL.md. Mecanicas, no de criterio."""
    if lectura["error"]:
        return [lectura["error"]]

    faltas = []
    meta = lectura["meta"]
    cuerpo = lectura["cuerpo"]
    plano_completo = _normalizar(lectura["texto"])

    for campo in ("name", "description", "version"):
        if campo not in meta or not meta[campo]:
            faltas.append(f"al frontmatter le falta el campo '{campo}'")

    for marca in _MARCADORES_DE_PLANTILLA:
        if marca in plano_completo:
            faltas.append(f"todavia tiene el texto de la plantilla ('{marca}...'): "
                          f"eso hay que reemplazarlo por lo suyo")
            break

    descripcion = meta.get("description", "")
    if descripcion:
        palabras = len(_normalizar(descripcion).split())
        if palabras < 8:
            faltas.append(f"la 'description' tiene {palabras} palabras: con eso la herramienta "
                          f"no puede decidir cuando disparar el skill. Diga que sale y cuando "
                          f"se usa")
        if _normalizar(descripcion).strip() == _normalizar(meta.get("name", "")).strip():
            faltas.append("la 'description' repite el 'name': no agrega informacion")

    instrucciones = cuerpo
    bloque_formato = _bloque_de_seccion(cuerpo, "Formato")
    if bloque_formato is not None:
        instrucciones = cuerpo.split(bloque_formato)[0]
    utiles = [linea for linea in instrucciones.splitlines()
              if linea.strip() and not linea.strip().startswith("#")]
    if len(utiles) < 2:
        faltas.append("no hay instrucciones: entre el frontmatter y el bloque de formato solo "
                      "esta el titulo. El modelo va a improvisar el contenido")

    if bloque_formato is None:
        faltas.append("falta el bloque '## Formato': sin el, la salida es distinta cada vez "
                      "que ejecute el skill")
    else:
        secciones = secciones_de_formato(cuerpo)
        lineas_utiles = [l for l in bloque_formato.splitlines() if l.strip()]
        if len(secciones) < 2 and len(lineas_utiles) < 4:
            faltas.append(f"el bloque '## Formato' tiene {len(lineas_utiles)} lineas con "
                          f"contenido: eso no describe una salida, la insinua. Escriba las "
                          f"secciones exactas con '### '")

    reglas = reglas_declaradas(cuerpo)
    if _bloque_de_seccion(cuerpo, "Reglas") is None:
        faltas.append("falta el bloque '## Reglas': sin reglas el modelo repite sus errores "
                      "por defecto")
    elif len(reglas) < 2:
        faltas.append(f"el bloque '## Reglas' tiene {len(reglas)} regla(s): el minimo del curso "
                      f"son dos")
    if reglas:
        limites = [r for r in reglas
                   if re.search(r"\bno\b|\bnunca\b|\bjamas\b|\bmaximo\b|\bsolo\b|\bunicamente\b",
                                _normalizar(r))]
        if not limites:
            faltas.append("ninguna regla pone un limite de alcance: falta al menos una que diga "
                          "que NO debe hacer el skill")
    return faltas


def faltas_de_anclaje(lectura, columnas):
    """Comprueba que el skill nombre columnas reales del proyecto. Detecta lo generico."""
    if lectura["error"]:
        return [lectura["error"]], []
    reales = [c for c in columnas if str(c).strip()
              and "escriba" not in _normalizar(c) and "columna" != _normalizar(c).strip()]
    if len(reales) < 2:
        return (["todavia no escribio sus columnas reales: llene la lista de la Tarea 0 con los "
                 "nombres exactos de su CSV"], [])
    plano = _normalizar(lectura["texto"])
    nombradas = [c for c in reales if _normalizar(c).strip() and _normalizar(c).strip() in plano]
    if len(nombradas) < 2:
        return ([f"el skill nombra {len(nombradas)} de sus {len(reales)} columnas reales: asi "
                 f"serviria igual para cualquier dataset del mundo, y los skills genericos ya "
                 f"existen. Nombre sus columnas, sus umbrales y sus preguntas"], nombradas)
    return [], nombradas


def banderas_de_seguridad(texto):
    """Las banderas rojas 1, 2 y 3 del criterio del curso. Las 4 y 5 no son detectables aqui."""
    plano = _normalizar(texto)
    crudo = str(texto).lower()
    banderas = []

    sensibles = ("env", "credentials", "credenciales", "id rsa", "ssh", "netrc",
                 "bash history", "zsh history", "secrets", "keychain", "aws")
    encontrados = sorted({s for s in sensibles if re.search(r"\b" + s.replace(" ", r"[ _.]") + r"\b",
                                                            plano)})
    if encontrados:
        banderas.append(("1 · archivos sensibles",
                         f"el skill nombra {', '.join(encontrados)}. Un skill de analisis de "
                         f"datos no tiene por que leer eso"))

    urls = re.findall(r"https?://[^\s)\"']+", crudo)
    verbos = ("enviar", "envia", "envie", "manda", "mandar", "subir", "sube", "publicar",
              "reportar", "telemetria", "endpoint", "collect", "upload", "post ")
    if urls and any(v in plano for v in verbos):
        banderas.append(("2 · exfiltracion",
                         f"instruye mover contenido a una direccion externa: {urls[0]}. "
                         f"Su dataset y su codigo salen de su maquina"))

    credenciales = ("api key", "api_key", "apikey", "token", "contrasena", "password", "llave")
    pedidos = sorted({c for c in credenciales if c in plano or c in crudo})
    if pedidos:
        banderas.append(("3 · credenciales",
                         f"menciona {', '.join(pedidos)}. Un skill que pide secretos para "
                         f"'funcionar mejor' no los necesita para funcionar"))

    desactivadores = ("no preguntes antes", "sin pedir confirmacion", "sin preguntar",
                      "se proactivo", "no pidas permiso", "no interrumpas")
    activos = sorted({d for d in desactivadores if d in plano})
    if activos:
        banderas.append(("extra · desactiva la confirmacion",
                         f"la regla '{activos[0]}' apaga justamente la pregunta que lo habria "
                         f"salvado. Una regla que le quita frenos al modelo no es una regla"))
    return banderas


def faltas_de_ecosistema(plan):
    """Forma del plan de tres skills, incluida la regla de las dos fases distintas."""
    faltas = []
    if not isinstance(plan, (list, tuple)) or len(plan) != 3:
        return ["el plan tiene que ser una lista de exactamente tres skills"]

    fases_validas = {"preparacion", "analisis", "comunicacion"}
    fases = []
    nombres = []
    salidas = []
    for i, fila in enumerate(plan, start=1):
        if not isinstance(fila, (list, tuple)) or len(fila) != 3:
            faltas.append(f"el skill {i} no trae los tres datos (nombre, fase, que sale)")
            continue
        nombre, fase, sale = (str(x).strip() for x in fila)
        nombres.append(nombre)
        fases.append(_normalizar(fase).strip())
        salidas.append(sale)
        if not re.fullmatch(r"[a-z0-9]+(-[a-z0-9]+)*", nombre):
            faltas.append(f"el nombre '{nombre}' no es un nombre de carpeta: minusculas, "
                          f"guiones, sin tildes ni espacios")
        if _normalizar(fase).strip() not in fases_validas:
            faltas.append(f"la fase del skill {i} es '{fase}': tiene que ser preparacion, "
                          f"analisis o comunicacion")
        if len(_normalizar(sale).split()) < 5:
            faltas.append(f"lo que sale del skill {i} cabe en menos de cinco palabras: si no "
                          f"puede describir la salida, el skill todavia no existe")
        if any(m in _normalizar(sale) for m in _MARCADORES_DE_PLANTILLA):
            faltas.append(f"lo que sale del skill {i} sigue siendo el texto de la plantilla")

    con_nombre = [n for n in nombres if n]
    if len(set(con_nombre)) < len(con_nombre):
        faltas.append("hay dos skills con el mismo nombre")
    distintas = {f for f in fases if f in fases_validas}
    if len(distintas) < 2 and not faltas:
        faltas.append(f"los tres skills cubren una sola fase ({', '.join(distintas)}): eso no es "
                      f"un ecosistema, es un skill partido en tres. Fusione dos y busque el "
                      f"tercero en otra fase")
    return faltas


def faltas_de_formato_previsto(cuerpo, salida_prevista):
    """Compara la salida que el estudiante espera contra el contrato de su propio skill."""
    secciones = secciones_de_formato(cuerpo)
    if not secciones:
        return (["su skill no declara secciones con '### ' dentro de '## Formato': sin contrato "
                 "no hay nada que comparar, y la salida va a cambiar en cada ejecucion"], [], [])
    if salida_prevista is None or not str(salida_prevista).strip():
        return (["todavia no escribio la salida que espera"], secciones, [])
    plano = _normalizar(salida_prevista)
    presentes = [s for s in secciones if _normalizar(s).strip() in plano]
    ausentes = [s for s in secciones if s not in presentes]
    faltas = []
    if ausentes:
        faltas.append("la salida que escribio no trae estas secciones que su propio skill "
                      "promete: " + "; ".join(ausentes))
    if len(_normalizar(salida_prevista).split()) < 25:
        faltas.append("la salida que escribio es demasiado corta para ser una salida: rellene "
                      "cada seccion con el contenido concreto que espera, con cifras inventadas "
                      "si hace falta")
    return faltas, secciones, presentes


# --- Envoltorios que imprimen y llevan el estado --------------------------

_AVISO_MECANICO = ("Esto es una revision MECANICA: mira la forma del archivo, no si el skill es "
                   "bueno. Eso lo decide usted leyendo la salida.")


def _reportar(clave, faltas, frase_ok, aviso=None):
    _RESULTADOS[clave] = False
    if faltas:
        print(f"[{clave}] Todavia no pasa:")
        for falta in faltas:
            print(f"[{clave}]   - {falta}")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
    else:
        _RESULTADOS[clave] = True
        print(f"[{clave}] {frase_ok}")
    if aviso:
        print(f"[{clave}] {aviso}")


def comprobar(clave, valor):
    """Dice si la respuesta es la correcta, sin revelar cual era."""
    _RESULTADOS[clave] = False
    if valor is None or (isinstance(valor, (list, tuple, set, str)) and not valor):
        print(f"[{clave}] Sin resolver todavia: la variable sigue vacia.")
        return
    if isinstance(valor, (list, tuple, set)):
        print(f"[{clave}] Usted respondio {len(valor)} elemento(s): "
              f"{sorted(str(v) for v in valor)}")
    else:
        print(f"[{clave}] Usted respondio: {valor!r}")
    if _huella(valor) == _ESPERADO.get(clave):
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


def revisar_skill(clave, ruta):
    """Revisa la FORMA de un SKILL.md: frontmatter, instrucciones, formato, reglas."""
    lectura = leer_skill(ruta)
    faltas = faltas_de_skill(lectura)
    print(f"[{clave}] Archivo: {ruta}")
    if not faltas:
        secciones = secciones_de_formato(lectura["cuerpo"])
        reglas = reglas_declaradas(lectura["cuerpo"])
        print(f"[{clave}] Declara {len(secciones)} seccion(es) de salida y "
              f"{len(reglas)} regla(s).")
    _reportar(clave, faltas,
              "La forma esta bien: frontmatter completo, instrucciones, formato y reglas.",
              _AVISO_MECANICO)


def revisar_anclaje(clave, ruta, columnas):
    """Comprueba que el skill nombre las columnas reales del proyecto del equipo."""
    lectura = leer_skill(ruta)
    faltas, nombradas = faltas_de_anclaje(lectura, columnas)
    if nombradas:
        print(f"[{clave}] Columnas suyas que el skill nombra: {', '.join(nombradas)}")
    _reportar(clave, faltas,
              "El skill esta anclado a su proyecto: nombra sus columnas reales.",
              "Nombrar columnas es lo minimo. Un skill anclado de verdad nombra tambien sus "
              "umbrales y sus preguntas de investigacion, y eso no lo puede medir un chequeo.")


def auditar_skill(clave, ruta=None, texto=None, veredicto=None):
    """Audita un SKILL.md contra las banderas rojas del curso.

    Detecta las banderas 1, 2 y 3. Las banderas 4 (descripcion vaga sobre lo que
    realmente hace) y 5 (autor desconocido, sin historial) NO son detectables por
    una maquina, y este chequeo lo dice cada vez. Ese es el punto: el criterio
    automatico encuentra lo evidente; leer el archivo completo es lo que encuentra
    el resto.
    """
    _RESULTADOS[clave] = False
    if texto is None:
        lectura = leer_skill(ruta)
        if lectura["error"]:
            print(f"[{clave}] No se pudo leer: {lectura['error']}")
            print(f"[{clave}] Pista: {_PISTAS[clave]}")
            return
        texto = lectura["texto"]
        print(f"[{clave}] Archivo: {ruta}")
    banderas = banderas_de_seguridad(texto)
    if banderas:
        print(f"[{clave}] Banderas rojas detectadas: {len(banderas)}")
        for etiqueta, detalle in banderas:
            print(f"[{clave}]   - Bandera {etiqueta}: {detalle}")
        print(f"[{clave}] Veredicto mecanico: NO INSTALAR.")
    else:
        print(f"[{clave}] Ninguna bandera roja de las detectables automaticamente.")
    print(f"[{clave}] Este chequeo NO puede juzgar las banderas 4 y 5: si la descripcion es vaga "
          f"sobre lo que el skill hace de verdad, y si el autor es alguien sin historial. "
          f"Esas dos se ven leyendo, y son las que mas se pasan por alto.")

    esperadas = _ESPERADO.get(clave)
    if esperadas is not None and _huella([e for e, _ in banderas]) != esperadas:
        print(f"[{clave}] AVISO: el conteo de banderas no es el que este archivo deberia dar. "
              f"Si edito el texto del skill, es normal.")
    if veredicto is None or not str(veredicto).strip():
        print(f"[{clave}] Falta lo unico que no puede hacer la maquina: su veredicto escrito. "
              f"Pase su decision y su razon como veredicto=\"...\".")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    palabras = len(_normalizar(veredicto).split())
    plano = _normalizar(veredicto)
    if palabras < 6:
        print(f"[{clave}] Su veredicto tiene {palabras} palabras: 'no instalar' no es un "
              f"veredicto, es una respuesta. Falta la razon.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    if not re.search(r"instalar|instalaria|confio|confiar|rechaz|descart", plano):
        print(f"[{clave}] Su veredicto no dice si lo instalaria o no. Decidalo: ante la duda, "
              f"no se instala.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    _RESULTADOS[clave] = True
    print(f"[{clave}] Veredicto registrado: \"{str(veredicto).strip()}\"")
    print(f"[{clave}] REGISTRADO. El curso no comprueba si su veredicto es el mismo del "
          f"profesor: comprueba que usted decidio y dijo por que.")


def comparar_formato(clave, ruta, salida_prevista):
    """Compara la salida que usted espera contra el contrato que su propio skill declara.

    Este es el chequeo que reemplaza a "ejecutelo y mire la salida". No necesita un
    modelo, y es mas exigente: le obliga a escribir primero que espera, y despues
    comprueba que su skill de verdad lo prometa. Si no coinciden, el que esta mal
    casi siempre es el skill.
    """
    lectura = leer_skill(ruta)
    if lectura["error"]:
        _reportar(clave, [lectura["error"]], "")
        return
    faltas, secciones, presentes = faltas_de_formato_previsto(lectura["cuerpo"], salida_prevista)
    if secciones:
        print(f"[{clave}] Su skill promete estas secciones: {'; '.join(secciones)}")
        print(f"[{clave}] Su salida prevista trae {len(presentes)} de {len(secciones)}.")
    _reportar(clave, faltas,
              "Su salida prevista cumple el contrato que declara su propio skill.",
              "Lo que esto comprueba es que su formato es especifico y usted lo respeta. Que la "
              "salida real del modelo se le parezca es lo que va a verificar cuando lo ejecute.")


def revisar_ecosistema(clave, plan):
    """Revisa la forma del plan de tres skills y la regla de las dos fases distintas."""
    faltas = faltas_de_ecosistema(plan)
    if isinstance(plan, (list, tuple)):
        for i, fila in enumerate(plan, start=1):
            if isinstance(fila, (list, tuple)) and len(fila) == 3:
                print(f"[{clave}]   {i}. {fila[0]}  [{fila[1]}]  -> {fila[2]}")
    _reportar(clave, faltas,
              "El plan tiene forma de ecosistema: tres skills distintos y al menos dos fases.",
              "Que sean los tres skills que su proyecto de verdad necesita no lo puede saber un "
              "chequeo. Eso se ve en la clase 12, cuando los use o no los use.")


def revision_de_node():
    """Informa si esta maquina tiene Node.js. NUNCA falla ni bloquea el cuaderno.

    Instalar Node.js y un CLI de IA es una RECOMENDACION de este curso, no un
    requisito. Todo lo que se evalua se hace sin eso.
    """
    import shutil
    ruta_node = shutil.which("node")
    ruta_npm = shutil.which("npm")
    if ruta_node and ruta_npm:
        print("Esta maquina tiene Node.js y npm instalados.")
        print(f"  node: {ruta_node}")
        print(f"  npm:  {ruta_npm}")
        print("Puede hacer la seccion opcional del final si quiere. Sigue siendo opcional.")
    else:
        print("Esta maquina no tiene Node.js y npm en el PATH, y no pasa nada.")
        print("La clase completa y el reto se hacen sin eso. La seccion opcional del final")
        print("no aplica hoy; si algun dia la quiere hacer, esta en INSTALACION.md, seccion 12.")
    return bool(ruta_node and ruta_npm)


def resumen_puntos_de_control():
    """Estado de los puntos de control del cuaderno."""
    print("Punto de control")
    print("-" * 46)
    for clave in _CLAVES:
        estado = "en verde" if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logrados = sum(1 for c in _CLAVES if _RESULTADOS.get(c))
    print("-" * 46)
    print(f"{logrados} de {len(_CLAVES)} en verde.")
    print("Los puntos de forma son mecanicos: verde significa 'el archivo esta bien armado',")
    print("no significa 'el skill es bueno'. Esa parte la cierra usted, leyendo la salida.")


print("Verificador listo. Ninguna tarea de hoy se comprueba contra una respuesta correcta: lea la tabla de arriba.")

---

# Tarea 0 — Punto de partida

Antes de decidir qué skills necesita, escriba qué sabe de su proyecto. **Un skill anclado a un
proyecto que no está claro sale genérico**, y genérico es el error número uno de este reto.

Los nombres de columna que escriba aquí no son decorativos: el verificador los va a buscar dentro
de su primer skill. Escríbalos **exactamente como aparecen en su CSV**, con tildes y mayúsculas.

In [ ]:
# TU CÓDIGO AQUÍ
# Los datos de su proyecto. Las columnas van con el nombre exacto del CSV.

DATASET = ""          # nombre del archivo y de dónde salió
PREGUNTAS = [
    "",
    "",
    "",
]
COLUMNAS = [
    "",
    "",
    "",
    "",
]

# Las rutas de sus tres skills. Cambie los nombres cuando tenga el plan de la Tarea 1.
# Si usa otra herramienta, cambie ".gemini" por ".opencode", ".claude" o ".codex":
# el archivo de adentro es idéntico en las cuatro.
SKILL_1 = CARPETA_SKILLS / "skill-uno" / "SKILL.md"
SKILL_2 = CARPETA_SKILLS / "skill-dos" / "SKILL.md"
SKILL_3 = CARPETA_SKILLS / "skill-tres" / "SKILL.md"

for ruta in (SKILL_1, SKILL_2, SKILL_3):
    ruta.parent.mkdir(parents=True, exist_ok=True)
print("Columnas registradas:", [c for c in COLUMNAS if c])

**Su respuesta:** ¿qué parte del Momento 1 le tomó más tiempo del que valía?

_Escriba aquí:_

> Esa respuesta suele ser su primer skill.

**Su respuesta:** de lo que hicieron a mano en las clases 3, 4 y 5, ¿qué van a tener que volver a
hacer, igual, sobre el mismo dataset, de aquí a la clase 12?

_Escriba aquí:_

> Y esa es la definición operativa de "tarea repetitiva".

---

# Tarea 1 — El plan del ecosistema

Antes de escribir un solo archivo, decida **cuáles** tres skills necesita.

## Las fases

| Fase | Qué produce | Ejemplos de skill útil |
|------|-------------|------------------------|
| `preparacion` | Datos limpios y documentados | Reporte de calidad, generador de script de limpieza, diccionario de datos |
| `analisis` | Hallazgos y gráficos | Resumen de EDA, recomendador de tipo de gráfico, reportador de outliers |
| `comunicacion` | Dashboard, narrativa, README | Esquema de narrativa, redactor de resumen ejecutivo, generador de README |

**Regla del ecosistema:** los tres skills tienen que cubrir **al menos dos fases distintas**. Si
los tres son de preparación, fusione dos y busque el tercero en otra fase.

**Criterio de corte:** si no puede describir en una frase qué sale del skill, el skill todavía no
existe. Vuelva a pensarlo antes de abrir un archivo.

## Alineación con lo que viene

La clase 12 pide dashboard más narrativa. Un skill que arme el esqueleto de la narrativa, o que
pase la checklist de visualización de la clase 8, se paga solo en cinco semanas. Téngalo en cuenta
al elegir el tercero.

In [ ]:
# TU CÓDIGO AQUÍ
# Tres tuplas: (nombre-en-formato-de-carpeta, fase, qué sale de él).
# La fase es una de: preparacion, analisis, comunicacion.

PLAN = [
    ("", "", ""),
    ("", "", ""),
    ("", "", ""),
]

revisar_ecosistema("R1", PLAN)

**Su respuesta:** ¿por qué el skill 1 es el más prioritario y no otro?

_Escriba aquí:_

**Su respuesta:** ¿cuál de los tres va a usar más veces de aquí a la clase 12? No tiene por qué
ser el mismo que el prioritario.

_Escriba aquí:_

---

# Tarea 2 — Construir los tres skills

Vuelva a la celda de la Tarea 0 y **cambie los nombres de `SKILL_1`, `SKILL_2` y `SKILL_3`** por
los tres nombres de su plan. Después vuelva a ejecutarla.

## Las piezas, sin el orden

Todo lo que tiene que quedar dentro de cada archivo está en esta lista. **El orden no está dado a
propósito:** armarlo es parte del ejercicio, y ya vio dos archivos completos en el Bloque 2.

- Un bloque `## Reglas` con **al menos dos** viñetas.
- Los tres guiones de cierre del frontmatter, en su propia línea.
- Un campo `version`.
- Las instrucciones: qué tiene que hacer el skill, nombrando sus columnas reales.
- Un bloque `## Formato` con las secciones exactas de la salida, cada una con `### `.
- Los tres guiones de apertura del frontmatter, en la **línea 1** del archivo.
- Un campo `description` que diga **cuándo** se usa el skill, no solo qué es.
- Al menos una regla que diga algo que el skill **NO** debe hacer.
- Un campo `name`.
- Un título con `# ` debajo del frontmatter.

## Dos cosas que se olvidan siempre

1. **La `description` es lo que dispara el skill.** Si es vaga, la herramienta nunca sabe cuándo
   usarlo y usted cree que el skill no funciona.
2. **La sección de formato es lo que hace consistente la salida.** Si falta, el modelo adivina, y
   adivina distinto cada vez. Es el diagnóstico correcto el 90% de las veces cuando alguien se
   queja de "inconsistencia".

## Y una regla propia de este reto

Cada skill tiene que estar **anclado a su proyecto**: sus columnas reales, sus umbrales reales,
sus preguntas de investigación reales. Un skill que funcionaría igual para cualquier dataset es un
skill genérico, y los genéricos ya existen: no aporta nada escribiéndolos otra vez.

El verificador comprueba el anclaje del skill 1 contra las columnas de la Tarea 0.

## Skill 1

Antes de escribirlo, conteste esto. Sirve de andamiaje: lo que responda aquí es literalmente lo
que va en el archivo.

**Su respuesta:** ¿cuándo debería activarse este skill? (va en `description`)

_Escriba aquí:_

**Su respuesta:** ¿cuáles son las secciones exactas de la salida? (van en `## Formato`, con `### `)

_Escriba aquí:_

**Su respuesta:** ¿qué dos cosas NO debe hacer? (van en `## Reglas`)

_Escriba aquí:_

In [ ]:
%%writefile .gemini/skills/skill-uno/SKILL.md
---
name: [Nombre legible del skill]
description: [Una frase que diga cuando usar este skill y que sale de el]
version: 1.0.0
---

# [Nombre legible del skill]

# TU CÓDIGO AQUÍ
[Las instrucciones. Nombre las columnas reales de su dataset, sus umbrales y sus
preguntas de investigacion.]

## Formato

# TU CÓDIGO AQUÍ
### [Primera seccion de la salida]
[Que va adentro.]

### [Segunda seccion de la salida]
[Que va adentro.]

## Reglas

# TU CÓDIGO AQUÍ
- [Regla 1: una restriccion concreta]
- [Regla 2: un limite de alcance, algo que el skill NO debe hacer]

In [ ]:
revisar_skill("R2", SKILL_1)

## Skill 2

**Su respuesta:** ¿cuándo debería activarse este skill?

_Escriba aquí:_

**Su respuesta:** ¿cuáles son las secciones exactas de la salida?

_Escriba aquí:_

**Su respuesta:** ¿qué dos cosas NO debe hacer?

_Escriba aquí:_

In [ ]:
%%writefile .gemini/skills/skill-dos/SKILL.md
---
name: [Nombre legible del skill]
description: [Una frase que diga cuando usar este skill y que sale de el]
version: 1.0.0
---

# [Nombre legible del skill]

# TU CÓDIGO AQUÍ
[Las instrucciones. Nombre las columnas reales de su dataset, sus umbrales y sus
preguntas de investigacion.]

## Formato

# TU CÓDIGO AQUÍ
### [Primera seccion de la salida]
[Que va adentro.]

### [Segunda seccion de la salida]
[Que va adentro.]

## Reglas

# TU CÓDIGO AQUÍ
- [Regla 1: una restriccion concreta]
- [Regla 2: un limite de alcance, algo que el skill NO debe hacer]

In [ ]:
revisar_skill("R3", SKILL_2)

## Skill 3

Este es el que se puede quedar sin probar. **Sin escribir, no.**

**Su respuesta:** ¿cuándo debería activarse este skill?

_Escriba aquí:_

In [ ]:
%%writefile .gemini/skills/skill-tres/SKILL.md
---
name: [Nombre legible del skill]
description: [Una frase que diga cuando usar este skill y que sale de el]
version: 1.0.0
---

# [Nombre legible del skill]

# TU CÓDIGO AQUÍ
[Las instrucciones. Nombre las columnas reales de su dataset, sus umbrales y sus
preguntas de investigacion.]

## Formato

# TU CÓDIGO AQUÍ
### [Primera seccion de la salida]
[Que va adentro.]

### [Segunda seccion de la salida]
[Que va adentro.]

## Reglas

# TU CÓDIGO AQUÍ
- [Regla 1: una restriccion concreta]
- [Regla 2: un limite de alcance, algo que el skill NO debe hacer]

In [ ]:
revisar_skill("R4", SKILL_3)

## El anclaje

Aquí se comprueba el error número uno de este reto. El verificador busca dentro de su skill 1 los
nombres de columna que escribió en la Tarea 0.

Si falla, la pregunta no es cómo pasar el chequeo: es **qué hace ese skill que no haría uno
genérico**. Si no hay respuesta, el skill sobra y hay que pensar otro.

In [ ]:
revisar_anclaje("R5", SKILL_1, COLUMNAS)

---

# Tarea 3 — La salida que usted espera

Esta es la tarea incómoda del reto, y es la que hace que un skill sirva.

**Escriba la salida que espera de su skill 1.** Completa, con las mismas secciones que su propio
bloque `## Formato` promete, con cifras inventadas donde haga falta pero con **sus** columnas y
**sus** categorías.

## Por qué se escribe en vez de ejecutarse

Cuando uno ejecuta un skill y mira la salida, está juzgando a posteriori: ya vio algo y decide si
le gusta. Es facilísimo conformarse. Cuando la escribe **antes**, tiene que decidir qué quiere, y
ahí se descubre que el bloque de formato no decía lo suficiente.

El verificador compara las dos cosas: las secciones que su skill **promete** contra las que su
salida esperada **trae**. Si no coinciden, el que está mal casi siempre es el skill, no la
predicción.

> Esta es también la mejor preparación posible para el día que sí ejecute el skill: va a tener
> contra qué comparar la salida real, en vez de leerla sin criterio.

In [ ]:
# TU CÓDIGO AQUÍ
# La salida que espera de su skill 1, con las secciones que él promete.
salida_prevista = """
"""

comparar_formato("R6", SKILL_1, salida_prevista)

**Su respuesta:** al escribir la salida esperada, ¿tuvo que volver a cambiar algo del
archivo? ¿Qué?

_Escriba aquí:_

> Si la respuesta es "no", vale la pena releer las dos con calma. Casi nunca es "no" la primera
> vez.

---

# Tarea 4 — Auditar lo que acaba de escribir

Los tres archivos que acaba de escribir son skills de terceros **para cualquiera que no sea
usted**: su compañero de equipo, el próximo semestre, quien los encuentre en su repositorio.

Páselos por el mismo auditor de la Parte 3 del Bloque 2. Toma treinta segundos y le enseña algo
sobre su propio texto: las instrucciones que le pidió leer archivos que no necesitaba, o las
reglas que le quitan frenos al modelo en vez de ponérselos.

Recuerde el límite del auditor: detecta las banderas 1, 2 y 3. Las banderas 4 (descripción vaga
sobre lo que el skill hace de verdad) y 5 (autor sin historial) **no las puede ver una máquina**.

In [ ]:
for etiqueta, ruta in (("skill 1", SKILL_1), ("skill 2", SKILL_2), ("skill 3", SKILL_3)):
    print(f"--- {etiqueta} ---")
    lectura = leer_skill(ruta)
    if lectura["error"]:
        print("  no se pudo leer:", lectura["error"])
    else:
        banderas = banderas_de_seguridad(lectura["texto"])
        print("  banderas rojas detectadas:", len(banderas))
        for etiqueta_bandera, detalle in banderas:
            print(f"    - {etiqueta_bandera}: {detalle}")

In [ ]:
# TU CÓDIGO AQUÍ
# El veredicto sobre su propio skill 1: si un compañero de otro equipo se lo encontrara,
# ¿lo instalaría? Diga sí o no, y por qué.
auditar_skill("R7", ruta=SKILL_1, veredicto="")

In [ ]:
resumen_puntos_de_control()

---

# Tarea 5 — Reflexión

Cortas y honestas. No hay respuesta correcta, y una respuesta crítica vale igual que una
entusiasta.

**Su respuesta:** ¿qué le sorprendió de escribir skills hoy?

_Escriba aquí:_

**Su respuesta:** ¿qué resultó más difícil de lo que esperaba?

_Escriba aquí:_

**Su respuesta:** de sus tres skills, ¿cuál le va a ahorrar más trabajo de aquí a la clase 12, y
por qué?

_Escriba aquí:_

**Su respuesta:** escribir la salida esperada obligó a decidir cosas que se venían posponiendo.
¿Cuál de esas decisiones fue la más incómoda?

_Escriba aquí:_

**Su respuesta (opcional):** ¿tiene alguna preocupación sobre usar IA en los entregables del
curso? Si cree que estas herramientas están sobrevaloradas, dígalo y explique por qué. Es una
respuesta perfectamente válida.

_Escriba aquí:_

---

# Cierre

## Qué entrega

Una carpeta comprimida `reto07_<apellidos>/` con:

- `reto_completo.ipynb` (este cuaderno, completo)
- `skills/` con los tres `SKILL.md`

**Fecha:** antes de la clase 8.

## Lo que viene: clase 8

**Principios de visualización.** Es la clase donde se aprende a juzgar un gráfico antes de
escribirlo. Uno de los skills más rentables que puede tener para el Momento 2 es justamente un
verificador de checklist de visualización: después de la clase 8 va a saber qué debe decir.

Guarde sus tres archivos. Son texto plano y siguen un estándar abierto: le van a servir en otras
materias y después de graduarse.

---

# Opcional — Ejecutar los skills de verdad

> **Recomendación del curso, no requisito.** No se evalúa, ni aquí ni en ninguna rúbrica.
> No la haga en clase si le quita tiempo a las cinco tareas de arriba.

Si su equipo tiene un CLI de IA instalado, o quiere instalarlo en casa, este es el cierre natural
de lo que hizo hoy: abrir la herramienta desde esta carpeta y pedirle que use su skill 1 sobre su
dataset real.

Las instrucciones de instalación están en [`../INSTALACION.md`](../INSTALACION.md), **sección 12**,
marcadas como opcionales.

Cuando tenga la salida real, el ejercicio que vale la pena es este, y es el mismo del Bloque 2:

1. Ponga la salida al lado de su `SKILL.md`.
2. Marque cada sección de la salida que **no** corresponde a lo que el formato prometía.
3. Marque cada regla que el modelo se saltó.
4. Cambie el archivo, no el prompt.
5. Vuelva a ejecutar.

Compare además la salida real con la que escribió en la Tarea 3. La distancia entre las dos es la
medida exacta de qué tan específico era su bloque de formato.

La celda de abajo solo mira si esta máquina tiene Node.js. **No instala nada y no falla nunca.**

In [ ]:
# SECCION OPCIONAL
# Solo informa. No instala nada y no falla nunca.
revision_de_node()